This script merges the demographic data (sometimes called integrated data) from 2014-2021 with the prescription data (from the same years) on the Observation ID variable. This generates the training dataset, which is used for cross-validation.

In [1]:
import pandas as pd

In [2]:
# Load integrated training data (2014-2021) and prescription data
df = pd.read_csv('data/integrated_data.csv')
rx = pd.read_csv('data/prescription.csv')

# Drop auto-generated index columns
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])
if 'Unnamed: 0' in rx.columns:
    rx = rx.drop(columns=['Unnamed: 0'])

print("Integrated data shape:", df.shape)
print("Prescription data shape:", rx.shape)

Integrated data shape: (904140, 10)
Prescription data shape: (904140, 9)


In [3]:
# Keep only suggested prescription features
rx = rx[['Observation_ID', 'Quantity', 'Form', 'Strength', 'Day_Supply']]

print("Prescription selected shape:", rx.shape)
print("Missing values:\n", rx.isnull().sum())

Prescription selected shape: (904140, 5)
Missing values:
 Observation_ID         0
Quantity           98066
Form               98066
Strength           98146
Day_Supply        330224
dtype: int64


In [4]:
# Join prescription features onto integrated data by Observation_ID
super_df = df.merge(rx, on='Observation_ID', how='left')

print("Super dataset shape:", super_df.shape)
print("Columns:", super_df.columns.tolist())
print("\nMissing values:\n", super_df.isnull().sum())

Super dataset shape: (904140, 14)
Columns: ['Observation_ID', 'Person_ID', 'Household_ID', 'Drug', 'Age', 'Sex', 'Family_income', 'Insurance_coverage', 'Race_ethnicity', 'Year', 'Quantity', 'Form', 'Strength', 'Day_Supply']

Missing values:
 Observation_ID             0
Person_ID                  0
Household_ID               0
Drug                       0
Age                     5261
Sex                        0
Family_income              0
Insurance_coverage         0
Race_ethnicity             0
Year                       0
Quantity               98066
Form                   98066
Strength               98146
Day_Supply            330224
dtype: int64


In [5]:
# Find which missing values need to be handled vs. which are from no prescriptions in prescription data
no_rx_mask = super_df['Drug'] == 'no prescriptions'

print(f"Total 'no prescriptions' rows: {no_rx_mask.sum()}")
print()
for col in ['Quantity', 'Form', 'Strength', 'Day_Supply']:
    missing_in_no_rx = super_df.loc[no_rx_mask, col].isnull().sum()
    missing_in_actual = super_df.loc[~no_rx_mask, col].isnull().sum()
    print(f"{col}:")
    print(f"  Missing in 'no prescriptions' rows: {missing_in_no_rx}")
    print(f"  Missing in actual drug rows:        {missing_in_actual}")

Total 'no prescriptions' rows: 98066

Quantity:
  Missing in 'no prescriptions' rows: 98066
  Missing in actual drug rows:        0
Form:
  Missing in 'no prescriptions' rows: 98066
  Missing in actual drug rows:        0
Strength:
  Missing in 'no prescriptions' rows: 98066
  Missing in actual drug rows:        80
Day_Supply:
  Missing in 'no prescriptions' rows: 98066
  Missing in actual drug rows:        232158


Missing values for Age and for the actual drug rows for Strength and Day_Supply are handled in each model to prevent data leakage in training and testing folds.

In [6]:
# Final summary of super dataset
print("FINAL SUPER DATASET SUMMARY")
print(f"Shape: {super_df.shape}")
print(f"\nColumns: {super_df.columns.tolist()}")
print(f"\nMissing values:")
print(super_df.isnull().sum())
print(f"\nUnique drugs: {super_df['Drug'].nunique()}")
print(f"No prescriptions rows: {(super_df['Drug'] == 'no prescriptions').sum()}")
print(f"Actual drug rows: {(super_df['Drug'] != 'no prescriptions').sum()}")
print(f"\nSample:")
print(super_df.head(3))

FINAL SUPER DATASET SUMMARY
Shape: (904140, 14)

Columns: ['Observation_ID', 'Person_ID', 'Household_ID', 'Drug', 'Age', 'Sex', 'Family_income', 'Insurance_coverage', 'Race_ethnicity', 'Year', 'Quantity', 'Form', 'Strength', 'Day_Supply']

Missing values:
Observation_ID             0
Person_ID                  0
Household_ID               0
Drug                       0
Age                     5261
Sex                        0
Family_income              0
Insurance_coverage         0
Race_ethnicity             0
Year                       0
Quantity               98066
Form                   98066
Strength               98146
Day_Supply            330224
dtype: int64

Unique drugs: 217
No prescriptions rows: 98066
Actual drug rows: 806074

Sample:
   Observation_ID   Person_ID  Household_ID        Drug   Age     Sex  \
0               1  4000110118       4000118  clobetasol  36.0    Male   
1               2  4007010318       4007018  clobetasol  33.0  Female   
2               3  40093

In [7]:
# Save super dataset
super_df.to_csv('data/super_integrated_data.csv', index=False)
print("Saved: super_integrated_data.csv")
print(f"Shape: {super_df.shape}")
print(f"\nFeatures:")
print("  Demographics: Age, Sex, Family_income, Insurance_coverage, Race_ethnicity")
print("  Prescription: Quantity, Form, Strength, Day_Supply")
print("  Target:       Drug")
print("  Identifier:   Observation_ID")

Saved: super_integrated_data.csv
Shape: (904140, 14)

Features:
  Demographics: Age, Sex, Family_income, Insurance_coverage, Race_ethnicity
  Prescription: Quantity, Form, Strength, Day_Supply
  Target:       Drug
  Identifier:   Observation_ID
